In [589]:
import requests
import json
import pandas as pd
from datetime import date
pd.set_option('display.max_rows', None)

# ID 112 is the Cubs 
team_id = 112
url = "https://statsapi.mlb.com/api/v1/schedule/games/?sportId=1"
team_queried = f"https://statsapi.mlb.com/api/v1/teams/{team_id}"
league = "https://statsapi.mlb.com/api/v1/teams/?sportId=1"
response = requests.get(team_queried)
data = response.json()
team_info = data['teams'][0]
team_name = ""
for key, value in team_info.items():
    if(key == "teamName"):
        team_name = value
        print(f"Team Name: {value}")

# Schedule variables
today = date.today()
schedule_url = f"https://statsapi.mlb.com/api/v1/schedule?teamId={team_id}&startDate=2026-03-10&endDate=2026-10-10&sportId=1"
schedule_response = requests.get(schedule_url)
schedule_data = schedule_response.json()


# Fill dataframe with game info from start of reuglar season to today
games = []
opponent_records = []
opponent_results = {}
dates = schedule_data.get('dates', [])
for day in dates:
    for game in day.get('games', []): 
        opponent = game.get('teams', {}).get('away' if game['teams']['home']['team']['id'] == team_id else 'home', {}).get('team', {}).get('name')
        winner = game['teams']['home'].get('isWinner') if game['teams']['home']['team']['id'] == team_id else game['teams']['away'].get('isWinner')
        wins = game['teams']['home']['leagueRecord'].get('wins') if game['teams']['home']['team']['id'] == team_id else game['teams']['away']['leagueRecord'].get('wins')
        losses = game['teams']['home']['leagueRecord'].get('losses') if game['teams']['home']['team']['id'] == team_id else game['teams']['away']['leagueRecord'].get('losses')
        team_percent = game['teams']['home']['leagueRecord'].get('pct') if game['teams']['home']['team']['id'] == team_id else game['teams']['away']['leagueRecord'].get('pct')
        opponent_percent = game['teams']['away']['leagueRecord'].get('pct') if game['teams']['home']['team']['id'] == team_id else game['teams']['home']['leagueRecord'].get('pct')

        # Create table of stats
        if game['gameType'] == "R":
            games.append({
                'Date': game.get('officialDate'),
                'Opponent': opponent,
                'Home/Away': 'Home' if game['teams']['home']['team']['id'] == team_id else 'Away',
                'Status': game.get('status', {}).get('detailedState'),
                'Winner': winner,
                # 'Record': f"{wins}-{losses}",
                'Percent': team_percent
            })
            opponent_records.append({
                'Opponent': opponent,
                'Opponent Record': opponent_percent
            })

            # Populate second table of head-to-head matchups
            if game['seriesDescription'] == "Regular Season":
                if opponent not in opponent_results:
                    opponent_results[opponent] = {"Team": opponent, "W": 0, "L": 0, "Future": 0}
                if winner == True:
                    opponent_results[opponent]["W"] += 1
                elif winner == False: 
                    opponent_results[opponent]["L"] += 1
                gameState = game.get('status', {}).get('abstractGameState')
                if (gameState != "Final") and (gameState != "Postponed"):
                    opponent_results[opponent]["Future"] += 1
    
    # Create table of head-to-head matchups
    vs_records = list(opponent_results.values())
    

# Display the schedule
if games:
    df = pd.DataFrame(games)
    df2 = pd.DataFrame(vs_records)
    df3 = pd.DataFrame(opponent_records)
    df2_sorted = df2.sort_values(by='Team')
    print("Team Schedule:")
    print(df)
    print('\n')
    print("Records Against Teams:")
    print(df2_sorted)
    print('\n')
    # print("Opponent's Records")
    # print(df3)
    #for result in opponent_results:
        #print(opponent_results[result]["Future"])
else:
    print("No scheduled games")



Team Name: Cubs
Team Schedule:
           Date               Opponent Home/Away     Status Winner Percent
0    2026-03-26   Washington Nationals      Home      Final  False    .000
1    2026-03-28   Washington Nationals      Home      Final   True    .500
2    2026-03-29   Washington Nationals      Home      Final  False    .333
3    2026-03-30     Los Angeles Angels      Home      Final   True    .500
4    2026-03-31     Los Angeles Angels      Home      Final  False    .400
5    2026-04-01     Los Angeles Angels      Home      Final   True    .500
6    2026-04-03    Cleveland Guardians      Away      Final  False    .429
7    2026-04-05    Cleveland Guardians      Away  Postponed   None    .500
8    2026-04-05    Cleveland Guardians      Away      Final   True    .500
9    2026-04-05    Cleveland Guardians      Away      Final  False    .444
10   2026-04-06         Tampa Bay Rays      Away      Final  False    .400
11   2026-04-07         Tampa Bay Rays      Away      Final   True   

In [590]:
last_five = list(games)
last_five.copy()
temp_count = (len(last_five)- 5)
recents = {}

game_num = 0
# last_five.reverse()
# opponent_records.reverse()
for game in games:
    gameState = game.get('Status')
    if (gameState != "Final"):
        del opponent_records[game_num]
        del last_five[game_num]
        game_num -= 1
    game_num += 1
last_five.reverse()
opponent_records.reverse()

# Show All Games
# for num in range(len(last_five)):
# Show Last 5 Games
for num in range(5):
    print(f"Game {num+1}: {last_five[num]}")
    print(f"Opponent Record at Game {num+1}: {opponent_records[num]}")

Game 1: {'Date': '2026-04-08', 'Opponent': 'Tampa Bay Rays', 'Home/Away': 'Away', 'Status': 'Final', 'Winner': True, 'Percent': '.500'}
Opponent Record at Game 1: {'Opponent': 'Tampa Bay Rays', 'Opponent Record': '.417'}
Game 2: {'Date': '2026-04-07', 'Opponent': 'Tampa Bay Rays', 'Home/Away': 'Away', 'Status': 'Final', 'Winner': True, 'Percent': '.455'}
Opponent Record at Game 2: {'Opponent': 'Tampa Bay Rays', 'Opponent Record': '.455'}
Game 3: {'Date': '2026-04-06', 'Opponent': 'Tampa Bay Rays', 'Home/Away': 'Away', 'Status': 'Final', 'Winner': False, 'Percent': '.400'}
Opponent Record at Game 3: {'Opponent': 'Tampa Bay Rays', 'Opponent Record': '.500'}
Game 4: {'Date': '2026-04-05', 'Opponent': 'Cleveland Guardians', 'Home/Away': 'Away', 'Status': 'Final', 'Winner': False, 'Percent': '.444'}
Opponent Record at Game 4: {'Opponent': 'Cleveland Guardians', 'Opponent Record': '.600'}
Game 5: {'Date': '2026-04-05', 'Opponent': 'Cleveland Guardians', 'Home/Away': 'Away', 'Status': 'Final'

In [591]:
# 3x + 2.0a + 1.8b + 1.6c + 1.4d + 1.2e
# x = team record and a-e are the last 5 games
# note - maybe multiply last 5 games by opposing team's record to make better opponents worth more to a team's recent strength
# potentially replace a-e with opponent's record from game day
# cubs would be 3(.500) + 2.0(-1) + 1.8(1) + 1.6(1) + 1.4(-1) + 1.2(1) as of 4/9/26 WITHOUT opponent weighting
# cubs would be 3(.500) - 2.0(.417) + 1.8(.455) + 1.6(.500) - 1.4(.600) + 1.2(.556) as of 4/9/26 WITH opponent weighting
only_records = []
for num in opponent_records:
    only_records.append([num['Opponent Record']])
for num in range(5):
    print(f"Opponent Record at Game {num+1}: {opponent_records[num]}")
print('\n')
print(f"Formula: 3({game.get('Percent')}) + 2.0({only_records[0][0]}) + 1.8({only_records[1][0]}) + 1.6({only_records[2][0]}) + 1.4({only_records[3][0]}) + 1.2({only_records[4][0]})")
print(f"Recent Weighted Strength of {team_name}: ", ((3*(float(game.get('Percent')))) + (2*(float(only_records[0][0])))
                                            +(1.8*(float(only_records[1][0]))) + (1.6*(float(only_records[2][0])))
                                            + (1.4*(float(only_records[3][0]))) + (1.2*(float(only_records[4][0])))))

Opponent Record at Game 1: {'Opponent': 'Tampa Bay Rays', 'Opponent Record': '.417'}
Opponent Record at Game 2: {'Opponent': 'Tampa Bay Rays', 'Opponent Record': '.455'}
Opponent Record at Game 3: {'Opponent': 'Tampa Bay Rays', 'Opponent Record': '.500'}
Opponent Record at Game 4: {'Opponent': 'Cleveland Guardians', 'Opponent Record': '.600'}
Opponent Record at Game 5: {'Opponent': 'Cleveland Guardians', 'Opponent Record': '.556'}


Formula: 3(.500) + 2.0(.417) + 1.8(.455) + 1.6(.500) + 1.4(.600) + 1.2(.556)
Recent Weighted Strength of Cubs:  5.4602


In [592]:
from datetime import datetime

cubs_id = 112
current_date = datetime.now()

past_home_wins = []
future_home_games = []

for date_entry in schedule_data["dates"]:
    for game in date_entry["games"]:
        game_date = datetime.fromisoformat(game["gameDate"].replace("Z", "+00:00"))
        home_team = game["teams"]["home"]["team"]["id"]
        away_team = game["teams"]["away"]["team"]["id"]
        home_score = game["teams"]["home"].get("score", 0)
        away_score = game["teams"]["away"].get("score", 0)

        is_cubs_home = home_team == cubs_id
        is_cubs_away = away_team == cubs_id

        if is_cubs_home:
            current_date = datetime.now().replace(tzinfo=None)
            game_date = game_date.replace(tzinfo=None)
            if game_date < current_date:
                # Past game
                if is_cubs_home and home_score >= 1:
                    past_home_wins.append({**game, "isCubsHome": True})
                elif is_cubs_away and away_score >= 1:
                    past_home_wins.append({**game, "isCubsHome": False})
            else:
                # Future game
                future_home_games.append(game)

# print(future_home_games)